# 🎬 CineIQ - Explainable Hybrid Movie Recommendation System

This notebook demonstrates the complete CineIQ pipeline:
1. **Data Loading & Preprocessing** - Merging MovieLens, TMDB, and IMDB datasets
2. **Content-Based Filtering** - TF-IDF vectorization + cosine similarity
3. **Collaborative Filtering** - SVD matrix factorization (Surprise library)
4. **Hybrid Ensemble** - Weighted combination (40% Content + 60% CF)
5. **Sentiment Analysis** - VADER sentiment scoring on movie overviews
6. **Explainability** - LIME + rule-based explanations
7. **User Taste Profiling** - Genre radar charts, decade preferences, actor/director affinities

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity

print('All imports successful ✅')

## 2. Data Loading & Exploration

In [ ]:
# Load the processed merged dataset
movies_df = pd.read_parquet('../data/processed/movies_merged.parquet')
print(f'Loaded {len(movies_df)} movies')
print(f'Columns: {movies_df.columns.tolist()}')
movies_df.head()

In [ ]:
# Data quality overview
print('=== Dataset Overview ===')
print(f'Total movies: {len(movies_df)}')
print(f'Movies with TMDB metadata: {movies_df["cast"].notna().sum()}')
print(f'Movies with directors: {movies_df["director"].notna().sum()}')
print(f'Movies with keywords: {movies_df["keywords"].notna().sum()}')
print(f'Movies with overviews: {movies_df["overview"].notna().sum()}')
print()
print('=== Missing Values ===')
print(movies_df.isnull().sum())

In [ ]:
# Genre distribution
all_genres = []
for g in movies_df['genres'].dropna():
    all_genres.extend([x.strip() for x in str(g).split('|') if x.strip() and x.strip() != '(no genres listed)'])

genre_counts = pd.Series(all_genres).value_counts().head(15)

fig = px.bar(
    x=genre_counts.values, y=genre_counts.index,
    orientation='h',
    labels={'x': 'Count', 'y': 'Genre'},
    title='Top 15 Movie Genres in Dataset',
    color=genre_counts.values,
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False, coloraxis_showscale=False)
fig.show()

## 3. Content-Based Filtering (TF-IDF + Cosine Similarity)

In [ ]:
# Create content 'soup' for TF-IDF
def create_soup(row):
    parts = []
    for col in ['genres', 'cast', 'keywords', 'overview', 'director']:
        if col in row and pd.notnull(row[col]):
            val = str(row[col]).replace('|', ' ').replace(',', ' ')
            parts.append(val)
    return ' '.join(parts)

movies_df['soup'] = movies_df.apply(create_soup, axis=1)

# Fit TF-IDF
tfidf = TfidfVectorizer(stop_words='english', max_features=20000)
tfidf_matrix = tfidf.fit_transform(movies_df['soup'])

print(f'TF-IDF Matrix Shape: {tfidf_matrix.shape}')
print(f'Vocabulary Size: {len(tfidf.vocabulary_)}')

In [ ]:
# Content-based recommendation function
def get_content_recommendations(title, top_n=10):
    idx_list = movies_df[movies_df['title'].str.lower() == title.lower()].index
    if len(idx_list) == 0:
        matched = movies_df[movies_df['title'].str.contains(title, case=False, regex=False, na=False)]
        if matched.empty:
            print(f"Movie '{title}' not found.")
            return pd.DataFrame()
        idx = matched.index[0]
    else:
        idx = idx_list[0]
    
    cosine_sim = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    sim_indices = cosine_sim.argsort()[:-(top_n+2):-1]
    sim_indices = [i for i in sim_indices if i != idx][:top_n]
    
    results = movies_df.iloc[sim_indices][['title', 'genres', 'cast', 'director']].copy()
    results['similarity_score'] = [cosine_sim[i] for i in sim_indices]
    return results

# Test: Find movies similar to Toy Story
print('🎯 Content-Based Recommendations for "Toy Story (1995)":')
get_content_recommendations('Toy Story (1995)')

In [ ]:
# Test with another movie
print('🎯 Content-Based Recommendations for "The Dark Knight (2008)":')
get_content_recommendations('The Dark Knight (2008)')

## 4. Collaborative Filtering (SVD)

In [ ]:
# Load ratings data (sample for notebook speed)
ratings_df = pd.read_csv('../data/raw/ratings.csv')
print(f'Total ratings: {len(ratings_df):,}')
print(f'Unique users: {ratings_df["userId"].nunique():,}')
print(f'Unique movies: {ratings_df["movieId"].nunique():,}')
print(f'Rating range: {ratings_df["rating"].min()} - {ratings_df["rating"].max()}')

# Rating distribution
fig = px.histogram(
    ratings_df.sample(100000), x='rating', nbins=10,
    title='Rating Distribution (100K Sample)',
    color_discrete_sequence=['#E50914']
)
fig.show()

In [ ]:
# Train SVD on a sample
try:
    from surprise import Dataset, Reader, SVD, accuracy
    from surprise.model_selection import train_test_split, cross_validate
    
    # Sample for notebook speed
    sample_ratings = ratings_df.sample(n=200000, random_state=42)
    
    reader = Reader(rating_scale=(0.5, 5.0))
    data = Dataset.load_from_df(sample_ratings[['userId', 'movieId', 'rating']], reader)
    trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
    
    # Train SVD
    svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, random_state=42)
    svd.fit(trainset)
    
    # Evaluate
    predictions = svd.test(testset)
    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)
    
    print(f'SVD Results on 200K sample:')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  MAE:  {mae:.4f}')
    
except ImportError:
    print('scikit-surprise not installed. Install with: pip install scikit-surprise')
    svd = None

## 5. Hybrid Ensemble (40% Content + 60% CF)

In [ ]:
# Demonstrate the hybrid scoring mechanism
from models.recommender import HybridRecommender

rec = HybridRecommender()
rec.load_data()

# Content-based recommendations
content_recs = rec.get_content_similar('Inception (2010)', top_n=10)
print('Content-Based Top 10 for Inception:')
for i, r in enumerate(content_recs):
    print(f'  {i+1}. {r["title"]} (score: {r["score"]:.4f})')

## 6. Sentiment Analysis

In [ ]:
# Load IMDB reviews for sentiment analysis demo
imdb_df = pd.read_csv('../data/raw/imdb_reviews.csv')
print(f'IMDB Reviews: {len(imdb_df)}')
print(f'Columns: {imdb_df.columns.tolist()}')
print(f'\nSentiment Distribution:')
print(imdb_df['sentiment'].value_counts())

In [ ]:
# VADER Sentiment Analysis Demo
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    
    analyzer = SentimentIntensityAnalyzer()
    
    # Analyze sample reviews
    sample_reviews = imdb_df.sample(500, random_state=42)
    sample_reviews['vader_compound'] = sample_reviews['review'].apply(
        lambda x: analyzer.polarity_scores(str(x)[:1000])['compound']
    )
    sample_reviews['vader_label'] = sample_reviews['vader_compound'].apply(
        lambda x: 'positive' if x >= 0.05 else ('negative' if x <= -0.05 else 'neutral')
    )
    
    # Compare VADER predictions vs actual labels
    correct = (sample_reviews['vader_label'] == sample_reviews['sentiment']).sum()
    total = len(sample_reviews)
    vader_acc = correct / total
    print(f'VADER Accuracy on 500 samples: {vader_acc:.2%}')
    
    # Visualize VADER scores distribution
    fig = px.histogram(
        sample_reviews, x='vader_compound', color='sentiment',
        nbins=30, title='VADER Compound Score Distribution by True Sentiment',
        color_discrete_map={'positive': '#4CAF50', 'negative': '#E50914'}
    )
    fig.show()
    
except ImportError:
    print('vaderSentiment not installed. Install with: pip install vaderSentiment')

## 7. Explainability (Rule-Based + LIME)

In [ ]:
from models.explainer import Explainer

explainer = Explainer(recommender_instance=rec)

# Generate explanations for content-based recommendations
test_cases = [
    ('Toy Story 2 (1999)', 'Toy Story (1995)', 'content', 0.85),
    ('The Dark Knight Rises (2012)', 'The Dark Knight (2008)', 'hybrid', 0.72),
    ('Finding Nemo (2003)', 'Toy Story (1995)', 'content', 0.65),
]

print('=== Explainability Examples ===')
for rec_title, query_title, source, sentiment in test_cases:
    explanation = explainer.explain(
        recommended_title=rec_title,
        query_title=query_title,
        user_id=1,
        rec_source=source,
        sentiment_score=sentiment
    )
    print(f'\n📽️  Recommending: {rec_title}')
    print(f'    Because you liked: {query_title}')
    print(f'    💡 {explanation}')

## 8. User Taste Profiling

In [ ]:
# Generate taste profile for a user
profile = rec.get_user_taste_profile(user_id=1)
if profile:
    print(f'User {profile["userId"]} Taste Profile:')
    print(f'  Total Rated: {profile["total_rated"]}')
    print(f'  Highly Rated: {profile["highly_rated"]}')
    print(f'  Top Genres: {profile["top_genres"]}')
    print(f'  Top Decades: {profile["top_decades"]}')
    print(f'  Top Actors: {profile["top_actors"]}')
    print(f'  Top Directors: {profile["top_directors"]}')

In [ ]:
# Visualize user taste profile
if profile:
    # Genre Radar Chart
    genres = profile['top_genres']
    genre_dist = profile.get('genre_distribution', {})
    scores = [genre_dist.get(g, 1) for g in genres]
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'polar'}, {'type': 'bar'}]],
        subplot_titles=['Genre Radar', 'Decade Preferences']
    )
    
    # Radar chart
    fig.add_trace(
        go.Scatterpolar(
            r=scores + [scores[0]],
            theta=genres + [genres[0]],
            fill='toself',
            fillcolor='rgba(229, 9, 20, 0.3)',
            line=dict(color='#E50914', width=2),
            name='Genre Affinity'
        ),
        row=1, col=1
    )
    
    # Decade bar chart
    decades = profile['top_decades']
    decade_values = list(range(len(decades), 0, -1))
    fig.add_trace(
        go.Bar(
            x=decades, y=decade_values,
            marker_color='#E50914',
            name='Decade Affinity'
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title=f'User {profile["userId"]} Taste Profile',
        showlegend=False,
        height=400,
        template='plotly_dark'
    )
    fig.show()

## 9. End-to-End Pipeline Demo

In [ ]:
from models.sentiment_reranker import SentimentReRanker

# Full pipeline: Content → Sentiment Re-rank → Explain
reranker = SentimentReRanker(mode='vader')
reranker.load_data()

# Step 1: Get content recommendations
query_movie = 'Inception (2010)'
raw_recs = rec.get_content_similar(query_movie, top_n=10)

# Step 2: Sentiment re-ranking
reranked_df = reranker.rerank(raw_recs)

# Step 3: Add explanations
print(f'\n🎯 Full Pipeline Results for "{query_movie}":')
print('=' * 80)
for _, row in reranked_df.iterrows():
    explanation = explainer.explain(
        recommended_title=row['title'],
        query_title=query_movie,
        rec_source='content',
        sentiment_score=row.get('sentiment_score', 0.5)
    )
    print(f"\n📽️  {row['title']}")
    print(f"    Score: {row['final_score']:.4f} (Content: {row['score']:.4f}, Sentiment: {row['sentiment_score']:.4f})")
    print(f"    💡 {explanation}")

## 10. Summary

### Key Results:
- **Content-Based Filtering**: TF-IDF vectorization on ~62K movies with genres, cast, keywords, and plot overview
- **Collaborative Filtering**: SVD on MovieLens 25M ratings with tracked RMSE, Precision@5, Recall@5
- **Hybrid Ensemble**: 40% Content + 60% CF weighted scoring
- **Sentiment Re-Ranking**: VADER-based audience reception scoring with TMDB vote average blending
- **Explainability**: LIME perturbation + rule-based shared feature templates
- **Serving**: FastAPI REST API + Streamlit dashboard
- **Tracking**: MLflow experiment logging with hyperparameter sweeps

### Architecture:
```
User → Streamlit Dashboard → FastAPI Backend
                               ├── HybridRecommender (TF-IDF + SVD)
                               ├── SentimentReRanker (VADER / DistilBERT)
                               ├── Explainer (LIME + Rules)
                               └── MLflow Tracking
```